## Save and export the best model using joblib or pickle while ensuring model versioning, storage efficiency, and compatibility for deployment without using default serialization functions.

In [1]:
import os, struct, zlib, hashlib, json, time, subprocess, sys
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

ROOT = 'models'                                        
df = pd.read_csv('heart.csv')                         
features = [c for c in df.columns if c != 'target']
X, y = df[features].values.astype(float), df['target'].values

mean, std = X.mean(0), X.std(0)                       
clf = LogisticRegression(max_iter=1000).fit((X - mean) / std, y)

idx = np.random.default_rng(0).permutation(len(X)); accs = []   
for f in np.array_split(idx, 5):
    tr = np.setdiff1d(idx, f); m, s = X[tr].mean(0), X[tr].std(0)
    accs.append(np.mean(LogisticRegression(max_iter=1000).fit((X[tr] - m) / s, y[tr]).predict((X[f] - m) / s) == y[f]))
print('CV accuracy:', round(np.mean(accs), 3), '| features:', len(features))

CV accuracy: 0.822 | features: 13


In [2]:
params = dict(mean=mean, std=std, coef=clf.coef_[0], intercept=clf.intercept_)
for k, v in params.items(): print(f'{k:10s} shape {v.shape}')

mean       shape (13,)
std        shape (13,)
coef       shape (13,)
intercept  shape (1,)


In [3]:
MAGIC, FORMAT_VERSION = b'HMDL', 1

def save_model(path, params, version, dtype='float64', extra=None):
    raw = b''.join(np.asarray(params[k], dtype=dtype).tobytes() for k in ['mean', 'std', 'coef', 'intercept'])
    z = zlib.compress(raw, 9); use_z = len(z) < len(raw)                    
    payload = z if use_z else raw
    meta = dict(name='heart_lr', version=version, dtype=dtype, compressed=use_z, n_features=len(features), features=features,
                sha256=hashlib.sha256(payload).hexdigest(), created=time.strftime('%Y-%m-%d %H:%M:%S'),
                numpy=np.__version__, **(extra or {}))
    mb = json.dumps(meta, separators=(',', ':')).encode()
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, 'wb') as f:
        f.write(MAGIC + struct.pack('<HI', FORMAT_VERSION, len(mb)) + mb + payload)
    return meta

In [4]:
def parse(v): return tuple(int(i) for i in v.split('.'))
def bump(v, part):
    a, b, c = parse(v); return {'major': f'{a + 1}.0.0', 'minor': f'{a}.{b + 1}.0', 'patch': f'{a}.{b}.{c + 1}'}[part]

def registry():
    p = f'{ROOT}/registry.json'
    return json.load(open(p)) if os.path.exists(p) else {}

def register(version, dtype, extra):
    reg = registry(); path = f'{ROOT}/heart_lr/v{version}/model.hmdl'
    meta = save_model(path, params, version, dtype, extra)
    reg.setdefault('heart_lr', []).append(dict(version=version, file=path, dtype=dtype, bytes=os.path.getsize(path),
                                               sha256=meta['sha256'], cv_accuracy=extra['cv_accuracy']))
    json.dump(reg, open(f'{ROOT}/registry.json', 'w'), indent=1)
    return path

def latest(name='heart_lr'): return max(registry()[name], key=lambda r: parse(r['version']))

In [5]:
if os.path.exists(f'{ROOT}/registry.json'): os.remove(f'{ROOT}/registry.json')     
extra = dict(cv_accuracy=round(float(np.mean(accs)), 4), trained_rows=len(X), notes='Logistic Regression, standardized features')
v1 = register('1.0.0', 'float64', extra)
v2 = register(bump('1.0.0', 'minor'), 'float32', extra)
print(pd.DataFrame(registry()['heart_lr'])[['version', 'dtype', 'bytes', 'cv_accuracy', 'sha256']].assign(sha256=lambda d: d.sha256.str[:12]).to_string())
print('Latest version:', latest()['version'])

  version    dtype  bytes  cv_accuracy        sha256
0   1.0.0  float64    749       0.8217  5afb491e36d8
1   1.1.0  float32    589       0.8217  7cb366146ff2
Latest version: 1.1.0


In [6]:
LOADER = '''
import struct, zlib, hashlib, json
import numpy as np

MAGIC, SUPPORTED_FORMAT = b'HMDL', 1

class HeartModel:
    def __init__(self, path):
        b = open(path, 'rb').read()
        if b[:4] != MAGIC: raise ValueError('Not an HMDL file')
        fmt, n = struct.unpack('<HI', b[4:10])
        if fmt > SUPPORTED_FORMAT: raise ValueError(f'Format v{fmt} not supported, update loader')
        self.meta = json.loads(b[10:10 + n]); payload = b[10 + n:]
        if hashlib.sha256(payload).hexdigest() != self.meta['sha256']: raise ValueError('Checksum mismatch: file corrupted')
        raw = zlib.decompress(payload) if self.meta['compressed'] else payload
        a = np.frombuffer(raw, dtype=self.meta['dtype']).astype(float); k = self.meta['n_features']
        self.mean, self.std, self.coef, self.intercept = a[:k], a[k:2 * k], a[2 * k:3 * k], a[3 * k]

    def predict_proba(self, X):
        X = np.asarray(X, dtype=float)
        if X.shape[1] != self.meta['n_features']: raise ValueError(f"Expected {self.meta['n_features']} features")
        return 1 / (1 + np.exp(-(((X - self.mean) / self.std) @ self.coef + self.intercept)))

    def predict(self, X): return (self.predict_proba(X) > 0.5).astype(int)
'''
open(f'{ROOT}/hmdl_loader.py', 'w').write(LOADER)
sys.path.insert(0, ROOT)
from hmdl_loader import HeartModel
print('Loader file likhi:', f'{ROOT}/hmdl_loader.py')

Loader file likhi: models/hmdl_loader.py


In [7]:
ref = clf.predict_proba((X - mean) / std)[:, 1]
for r in registry()['heart_lr']:
    m = HeartModel(r['file']); p = m.predict_proba(X)
    print(f"v{r['version']} ({r['dtype']}): max |prob diff| = {np.abs(p - ref).max():.2e} | label mismatches = {int(np.sum(m.predict(X) != clf.predict((X - mean) / std)))} / {len(X)}")

v1.0.0 (float64): max |prob diff| = 0.00e+00 | label mismatches = 0 / 303
v1.1.0 (float32): max |prob diff| = 6.05e-08 | label mismatches = 0 / 303


In [8]:
# 8. Storage efficiency: file size = header + metadata + numbers
for r in registry()['heart_lr']:
    m = HeartModel(r['file']); k = m.meta['n_features']; payload = (3 * k + 1) * np.dtype(m.meta['dtype']).itemsize
    print(f"v{r['version']} {r['dtype']}: total {r['bytes']} B = header 10 + metadata {r['bytes'] - 10 - payload} + numbers {payload} | compressed={m.meta['compressed']}")

v1.0.0 float64: total 749 B = header 10 + metadata 419 + numbers 320 | compressed=False
v1.1.0 float32: total 589 B = header 10 + metadata 419 + numbers 160 | compressed=False


In [9]:
code = ("import sys, json; sys.path.insert(0, sys.argv[1]); from hmdl_loader import HeartModel; import numpy as np; "
        "m = HeartModel(sys.argv[2]); p = m.predict_proba(np.array([json.loads(sys.argv[3])]))[0]; "
        "print('sklearn imported:', 'sklearn' in sys.modules, '| version', m.meta['version'], '| prob row 0:', round(float(p), 6))")
print(subprocess.run([sys.executable, '-c', code, ROOT, latest()['file'], json.dumps(X[0].tolist())], capture_output=True, text=True).stdout.strip())
print('sklearn prob row 0 (notebook me):', round(float(ref[0]), 6))

bad = open(latest()['file'], 'rb').read(); bad = bad[:-1] + bytes([bad[-1] ^ 1]); open(f'{ROOT}/corrupt.hmdl', 'wb').write(bad)
try: HeartModel(f'{ROOT}/corrupt.hmdl')
except ValueError as e: print('Corrupt file:', e)

try: HeartModel(latest()['file']).predict_proba(X[:, :5])
except ValueError as e: print('Wrong shape:', e)
os.remove(f'{ROOT}/corrupt.hmdl')

sklearn imported: False | version 1.1.0 | prob row 0: 0.80046
sklearn prob row 0 (notebook me): 0.80046
Corrupt file: Checksum mismatch: file corrupted
Wrong shape: Expected 13 features
